In [1]:
pip install geopy

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
df_burn = pd.read_excel("/content/drive/MyDrive/Colab_Notebooks/Data/Competition_Data/Input/NIRD 20230130 Database_Hackathon.xlsx", sheet_name="Data Table NIRD 20230130")

## 1. Data Cleaning

### 1.1 Handling Missing Data
Replace NaN values with 0 for trauma and burn-related columns to ensure consistency.

In [5]:
def replace_nan_with_0(df_d):
  df_d.fillna(0, inplace=True)
  return df_d

In [6]:
replace_nan_with_0(df_burn['ADULT_TRAUMA_L1'])

,ADULT_TRAUMA_L1
0,0.0
1,0.0
2,0.0
3,1.0
4,0.0
...,...
630,0.0
631,0.0
632,0.0
633,0.0


In [7]:
replace_nan_with_0(df_burn['ADULT_TRAUMA_L2'])

,ADULT_TRAUMA_L2
0,1.0
1,1.0
2,1.0
3,0.0
4,0.0
...,...
630,0.0
631,1.0
632,1.0
633,1.0


In [8]:
replace_nan_with_0(df_burn['PEDS_TRAUMA_L1'])

,PEDS_TRAUMA_L1
0,0.0
1,0.0
2,0.0
3,0.0
4,1.0
...,...
630,0.0
631,0.0
632,0.0
633,0.0


In [9]:
replace_nan_with_0(df_burn['PEDS_TRAUMA_L2'])

,PEDS_TRAUMA_L2
0,1.0
1,1.0
2,0.0
3,0.0
4,0.0
...,...
630,0.0
631,0.0
632,0.0
633,0.0


In [10]:
replace_nan_with_0(df_burn['TRAUMA_ADULT'])

,TRAUMA_ADULT
0,1.0
1,1.0
2,1.0
3,1.0
4,0.0
...,...
630,0.0
631,1.0
632,1.0
633,1.0


In [11]:
replace_nan_with_0(df_burn['TRAUMA_PEDS'])

,TRAUMA_PEDS
0,1.0
1,1.0
2,0.0
3,0.0
4,1.0
...,...
630,0.0
631,0.0
632,0.0
633,0.0


In [12]:
replace_nan_with_0(df_burn['BURN_ADULT'])

,BURN_ADULT
0,0.0
1,0.0
2,0.0
3,1.0
4,0.0
...,...
630,1.0
631,0.0
632,0.0
633,0.0


In [13]:
replace_nan_with_0(df_burn['BURN_PEDS'])

,BURN_PEDS
0,0.0
1,0.0
2,0.0
3,0.0
4,1.0
...,...
630,1.0
631,0.0
632,0.0
633,0.0


## 2. Address Preparation
Combine the various components (`ADDRESS`, `CITY`, `STATE_FULL`, and `ZIPCODE`) into a single complete string. This is necessary to successfully geocode and retrieve exact coordinates (Longitude and Latitude).

In [14]:
df_burn['FULL_ADDRESS'] = df_burn['ADDRESS'] + ', ' + df_burn['CITY'] + ', ' + df_burn['STATE_FULL'] + ' ' + df_burn['ZIP_CODE'].astype(str)
df_burn.head()

,STATE_FULL,STATE,COUNTY,ADDRESS,CITY,ZIP_CODE,AHA_ID,HOSPITAL_NAME,TOTAL_BEDS,BURN_BEDS,...,ACS_VERIFIED,ADULT_TRAUMA_L1,ADULT_TRAUMA_L2,PEDS_TRAUMA_L1,PEDS_TRAUMA_L2,ABA_VERIFIED,TC_STATE_DESIGNATED,BC_STATE_DESIGNATED,PHONE,FULL_ADDRESS
0,Alaska,AK,Anchorage,4315 Diplomacy Dr,Anchorage,99508,6940010.0,Alaska Native Medical Center,173,NaN,...,Yes,0.0,1.0,0.0,1.0,NaN,Yes,NaN,(907) 563-2662,"4315 Diplomacy Dr, Anchorage, Alaska 99508"
1,Alaska,AK,Anchorage,3200 Providence Dr,Anchorage,99508,6940020.0,Providence Alaska Medical Center/Children's Ho...,401,NaN,...,Yes,0.0,1.0,0.0,1.0,NaN,Yes,NaN,(907) 562-2211,"3200 Providence Dr, Anchorage, Alaska 99508"
2,Alabama,AL,Houston,1108 Ross Clark Cir,Dothan,36301,6530373.0,Southeast Alabama Medical Center,387,NaN,...,No,0.0,1.0,0.0,0.0,NaN,Yes,NaN,(334) 793-8111,"1108 Ross Clark Cir, Dothan, Alabama 36301"
3,Alabama,AL,Jefferson,619 19th St S,Birmingham,35233,6530304.0,University of Alabama at Birmingham Hospital (...,1157,28.0,...,Yes,1.0,0.0,0.0,0.0,No,Yes,No,(205) 934-3411,"619 19th St S, Birmingham, Alabama 35233"
4,Alabama,AL,Jefferson,1600 7th Ave South,Birmingham,35233,6530170.0,Children's of Alabama (Children's of Alabama B...,351,6.0,...,No,0.0,0.0,1.0,0.0,No,Yes,No,(205) 638-9100,"1600 7th Ave South, Birmingham, Alabama 35233"


### 3. Geocode Hospital Locations
Using the various `FULL_ADDRESS` to get the `longitude` and `latitude` for those locations

In [15]:
import time
import os
from geopy.geocoders import Nominatim
print("Geolocator initialized.")

def geocode_and_cache(df, address="FULL_ADDRESS",cache_file="/content/drive/MyDrive/Colab_Notebooks/Data/Competition_Data/Output/Cache/address_geocoded.csv"):
  if os.path.exists(cache_file):
    print("Loading cached data ...")
    return pd.read_csv(cache_file)

  # Re-initialize geolocator with a longer timeout (e.g., 5 seconds)
  geolocator = Nominatim(user_agent="burn_locator", timeout=5)

  latitudes = []
  longitudes = []

  for address in df_burn['FULL_ADDRESS']:
      try:
          location = geolocator.geocode(address)
          if location:
              latitudes.append(location.latitude)
              longitudes.append(location.longitude)
          else:
              latitudes.append(None)
              longitudes.append(None)
      except Exception as e:
          print(f"Error geocoding {address}: {e}")
          latitudes.append(None)
          longitudes.append(None)
      time.sleep(2) # Keep the 2-second delay to respect usage policy

  df_burn['Latitude'] = latitudes
  df_burn['Longitude'] = longitudes

  df_burn.to_csv(cache_file, index=False)

  return df_burn

Geolocator initialized.


In [16]:
df_burn = geocode_and_cache(df_burn)

Loading cached data ...


In [17]:
df_burn.head()

,STATE_FULL,STATE,COUNTY,ADDRESS,CITY,ZIP_CODE,AHA_ID,HOSPITAL_NAME,TOTAL_BEDS,BURN_BEDS,...,ADULT_TRAUMA_L2,PEDS_TRAUMA_L1,PEDS_TRAUMA_L2,ABA_VERIFIED,TC_STATE_DESIGNATED,BC_STATE_DESIGNATED,PHONE,FULL_ADDRESS,Latitude,Longitude
0,Alaska,AK,Anchorage,4315 Diplomacy Dr,Anchorage,99508,6940010.0,Alaska Native Medical Center,173,NaN,...,1.0,0.0,1.0,NaN,Yes,NaN,(907) 563-2662,"4315 Diplomacy Dr, Anchorage, Alaska 99508",61.182772,-149.800177
1,Alaska,AK,Anchorage,3200 Providence Dr,Anchorage,99508,6940020.0,Providence Alaska Medical Center/Children's Ho...,401,NaN,...,1.0,0.0,1.0,NaN,Yes,NaN,(907) 562-2211,"3200 Providence Dr, Anchorage, Alaska 99508",61.187168,-149.819439
2,Alabama,AL,Houston,1108 Ross Clark Cir,Dothan,36301,6530373.0,Southeast Alabama Medical Center,387,NaN,...,1.0,0.0,0.0,NaN,Yes,NaN,(334) 793-8111,"1108 Ross Clark Cir, Dothan, Alabama 36301",31.214885,-85.361261
3,Alabama,AL,Jefferson,619 19th St S,Birmingham,35233,6530304.0,University of Alabama at Birmingham Hospital (...,1157,28.0,...,0.0,0.0,0.0,No,Yes,No,(205) 934-3411,"619 19th St S, Birmingham, Alabama 35233",33.506101,-86.801782
4,Alabama,AL,Jefferson,1600 7th Ave South,Birmingham,35233,6530170.0,Children's of Alabama (Children's of Alabama B...,351,6.0,...,0.0,1.0,0.0,No,Yes,No,(205) 638-9100,"1600 7th Ave South, Birmingham, Alabama 35233",33.504689,-86.805635


In [18]:
df_burn.dropna(subset=['Latitude', 'Longitude'], inplace=True)
print(f"Number of hospitals after dropping un-geocoded: {len(df_burn)}")
print(f"Nulls in Latitude after cleanup: {df_burn['Latitude'].isnull().sum()}")
print(f"Nulls in Longitude after cleanup: {df_burn['Longitude'].isnull().sum()}")
df_burn.head()

Number of hospitals after dropping un-geocoded: 588
Nulls in Latitude after cleanup: 0
Nulls in Longitude after cleanup: 0


,STATE_FULL,STATE,COUNTY,ADDRESS,CITY,ZIP_CODE,AHA_ID,HOSPITAL_NAME,TOTAL_BEDS,BURN_BEDS,...,ADULT_TRAUMA_L2,PEDS_TRAUMA_L1,PEDS_TRAUMA_L2,ABA_VERIFIED,TC_STATE_DESIGNATED,BC_STATE_DESIGNATED,PHONE,FULL_ADDRESS,Latitude,Longitude
0,Alaska,AK,Anchorage,4315 Diplomacy Dr,Anchorage,99508,6940010.0,Alaska Native Medical Center,173,NaN,...,1.0,0.0,1.0,NaN,Yes,NaN,(907) 563-2662,"4315 Diplomacy Dr, Anchorage, Alaska 99508",61.182772,-149.800177
1,Alaska,AK,Anchorage,3200 Providence Dr,Anchorage,99508,6940020.0,Providence Alaska Medical Center/Children's Ho...,401,NaN,...,1.0,0.0,1.0,NaN,Yes,NaN,(907) 562-2211,"3200 Providence Dr, Anchorage, Alaska 99508",61.187168,-149.819439
2,Alabama,AL,Houston,1108 Ross Clark Cir,Dothan,36301,6530373.0,Southeast Alabama Medical Center,387,NaN,...,1.0,0.0,0.0,NaN,Yes,NaN,(334) 793-8111,"1108 Ross Clark Cir, Dothan, Alabama 36301",31.214885,-85.361261
3,Alabama,AL,Jefferson,619 19th St S,Birmingham,35233,6530304.0,University of Alabama at Birmingham Hospital (...,1157,28.0,...,0.0,0.0,0.0,No,Yes,No,(205) 934-3411,"619 19th St S, Birmingham, Alabama 35233",33.506101,-86.801782
4,Alabama,AL,Jefferson,1600 7th Ave South,Birmingham,35233,6530170.0,Children's of Alabama (Children's of Alabama B...,351,6.0,...,0.0,1.0,0.0,No,Yes,No,(205) 638-9100,"1600 7th Ave South, Birmingham, Alabama 35233",33.504689,-86.805635


## 4. Data Segregation
Separate the dataset into two distinct dataframes: one for specialized Burn Centers and another for Non-Burn Centers.

In [19]:
df_non_burn_center = df_burn[(df_burn['BURN_ADULT']==0) & (df_burn['BURN_PEDS']==0)].copy()

In [20]:
df_burn_center = df_burn[(df_burn['BURN_ADULT']==1) | (df_burn['BURN_PEDS']== 1)].copy()

## 5. Initial Map Generation
Initialize an interactive Folium map to plot the spatial distribution of the geocoded burn centers.

In [21]:
pip install folium

In [22]:
import folium

# Initialize a Folium map centered around the average coordinates of the burn centers
# Using a default location if df_burn_center is empty or has no coordinates
if not df_burn_center.empty and 'Latitude' in df_burn_center.columns and 'Longitude' in df_burn_center.columns:
    map_center = [df_burn_center['Latitude'].mean(), df_burn_center['Longitude'].mean()]
else:
    map_center = [39.8283, -98.5795] # Geographic center of the contiguous United States

m = folium.Map(location=map_center, zoom_start=4)
print("Folium map initialized.")

Folium map initialized.


In [23]:
for index, row in df_burn_center.iterrows():
    # Create popup content with relevant information
    popup_content = f"<b>Hospital:</b> {row['HOSPITAL_NAME']}<br>"
    popup_content += f"<b>Address:</b> {row['FULL_ADDRESS']}<br>"
    popup_content += f"<b>Burn Beds:</b> {row['BURN_BEDS'] if pd.notna(row['BURN_BEDS']) else 'N/A'}<br>"
    popup_content += f"<b>ACS Verified:</b> {row['ACS_VERIFIED']}<br>"
    popup_content += f"<b>ABA Verified:</b> {row['ABA_VERIFIED']}"

    # Add marker to the map
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(popup_content, max_width=300),
        tooltip=row['HOSPITAL_NAME'] # Tooltip on hover
    ).add_to(m)

# Display the map
m

## 6. Distance Analysis and Visualization
Calculate the geodesic distances between non-burn centers and their closest specialized burn centers. Categorize these distances to understand potential resource sharing and patient transport needs (e.g., ground vs. air transport).

### 6.1 Proximity Category: Within 30 Miles
Identify non-burn centers within a 30-mile radius of a burn center, suitable for immediate ground transport and shared resources.

In [24]:
from geopy.distance import geodesic

print("geodesic imported successfully.")

geodesic imported successfully.


In [25]:
nearby_non_burn_centers_list = []

for index_non_burn, row_non_burn in df_non_burn_center.iterrows():
    non_burn_center_coords = (row_non_burn['Latitude'], row_non_burn['Longitude'])
    found_nearby = False
    for index_burn, row_burn in df_burn_center.iterrows():
        burn_center_coords = (row_burn['Latitude'], row_burn['Longitude'])

        # Ensure both coordinates are not None/NaN before calculating distance
        if pd.notna(non_burn_center_coords[0]) and pd.notna(non_burn_center_coords[1]) and \
           pd.notna(burn_center_coords[0]) and pd.notna(burn_center_coords[1]):
            distance_miles = geodesic(non_burn_center_coords, burn_center_coords).miles

            if distance_miles <= 30:
                nearby_non_burn_centers_list.append(row_non_burn)
                found_nearby = True
                break  # Found a nearby burn center, move to the next non-burn center

# Convert the list of Series (rows) into a DataFrame
df_nearby_non_burn_centers = pd.DataFrame(nearby_non_burn_centers_list)

print(f"Number of non-burn centers within 30 miles of a burn center: {len(df_nearby_non_burn_centers)}")
df_nearby_non_burn_centers.head()

Number of non-burn centers within 30 miles of a burn center: 267


,STATE_FULL,STATE,COUNTY,ADDRESS,CITY,ZIP_CODE,AHA_ID,HOSPITAL_NAME,TOTAL_BEDS,BURN_BEDS,...,ADULT_TRAUMA_L2,PEDS_TRAUMA_L1,PEDS_TRAUMA_L2,ABA_VERIFIED,TC_STATE_DESIGNATED,BC_STATE_DESIGNATED,PHONE,FULL_ADDRESS,Latitude,Longitude
9,Arizona,AZ,Maricopa,1955 W Frye Rd,Chandler,85224,6860028.0,Chandler Regional Medical Center,338,NaN,...,0.0,0.0,0.0,NaN,Yes,NaN,(480) 728-3000,"1955 W Frye Rd, Chandler, Arizona 85224",33.296848,-111.874084
10,Arizona,AZ,Maricopa,5555 W Thunderbird Rd,Glendale,85306,6860013.0,Banner Thunderbird Medical Center,595,NaN,...,1.0,0.0,0.0,NaN,Yes,NaN,(602) 865-5555,"5555 W Thunderbird Rd, Glendale, Arizona 85306",33.608948,-112.179739
11,Arizona,AZ,Maricopa,13677 W McDowell Rd,Goodyear,85395,6860027.0,Abrazo West Campus,188,NaN,...,1.0,0.0,0.0,NaN,Yes,NaN,(623) 882-1500,"13677 W McDowell Rd, Goodyear, Arizona 85395",33.461962,-112.352849
12,Arizona,AZ,Maricopa,1400 S Dobson Rd,Mesa,85202,6860016.0,Banner Desert Medical Center/Banner Children's...,605,NaN,...,1.0,0.0,1.0,NaN,Yes,NaN,(480) 412-3000,"1400 S Dobson Rd, Mesa, Arizona 85202",33.390213,-111.878137
13,Arizona,AZ,Maricopa,1111 E McDowell Rd,Phoenix,85006,6860250.0,Banner University Medical Center - Phoenix,698,NaN,...,0.0,0.0,0.0,NaN,Yes,NaN,(602) 239-2000,"1111 E McDowell Rd, Phoenix, Arizona 85006",33.464989,-112.057618


In [26]:
import folium

# Initialize a Folium map centered around the average coordinates of the burn centers
# Using a default location if df_burn_center is empty or has no coordinates
if not df_burn_center.empty and 'Latitude' in df_burn_center.columns and 'Longitude' in df_burn_center.columns:
    map_center = [df_burn_center['Latitude'].mean(), df_burn_center['Longitude'].mean()]
else:
    map_center = [39.8283, -98.5795] # Geographic center of the contiguous United States

m = folium.Map(location=map_center, zoom_start=4)
print("Folium map initialized.")

Folium map initialized.


In [27]:
for index, row in df_burn_center.iterrows():
    # Create popup content for burn centers
    popup_content = f"<b>Burn Center:</b> {row['HOSPITAL_NAME']}<br>"
    popup_content += f"<b>Address:</b> {row['FULL_ADDRESS']}<br>"
    popup_content += f"<b>Burn Beds:</b> {row['BURN_BEDS'] if pd.notna(row['BURN_BEDS']) else 'N/A'}<br>"
    popup_content += f"<b>ACS Verified:</b> {row['ACS_VERIFIED']}<br>"
    popup_content += f"<b>ABA Verified:</b> {row['ABA_VERIFIED']}"

    # Add marker for burn center
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(popup_content, max_width=300),
        tooltip=row['HOSPITAL_NAME'],
        icon=folium.Icon(color='red', icon='fire', prefix='fa') # Distinct icon for burn centers
    ).add_to(m)

    # Add a 30-mile radius circle around burn centers (1 mile = 1609.34 meters)
    folium.Circle(
        location=[row['Latitude'], row['Longitude']],
        radius=30 * 1609.34, # Radius in meters
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.1
    ).add_to(m)


for index, row in df_nearby_non_burn_centers.iterrows():
    # Create popup content for nearby non-burn centers
    popup_content = f"<b>Non-Burn Center:</b> {row['HOSPITAL_NAME']}<br>"
    popup_content += f"<b>Address:</b> {row['FULL_ADDRESS']}"

    # Add marker for nearby non-burn center with a different color
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(popup_content, max_width=300),
        tooltip=row['HOSPITAL_NAME'],
        icon=folium.Icon(color='blue', icon='hospital', prefix='fa') # Distinct icon for non-burn centers
    ).add_to(m)

# Display the map
m

### 6.2 Proximity Category: 30 to 150 Miles
Identify non-burn centers between 30 and 150 miles from the nearest burn center, generally ideal for air transportation.

In [28]:
non_burn_centers_30_150_miles_list = []

for index_non_burn, row_non_burn in df_non_burn_center.iterrows():
    non_burn_center_coords = (row_non_burn['Latitude'], row_non_burn['Longitude'])
    min_distance_to_burn_center = float('inf') # Initialize with a very large distance

    # Skip if non-burn center coordinates are not valid
    if pd.isna(non_burn_center_coords[0]) or pd.isna(non_burn_center_coords[1]):
        continue

    for index_burn, row_burn in df_burn_center.iterrows():
        burn_center_coords = (row_burn['Latitude'], row_burn['Longitude'])

        # Ensure both coordinates are not None/NaN before calculating distance
        if pd.notna(burn_center_coords[0]) and pd.notna(burn_center_coords[1]):
            distance_miles = geodesic(non_burn_center_coords, burn_center_coords).miles
            min_distance_to_burn_center = min(min_distance_to_burn_center, distance_miles)

    # If the minimum distance to *any* burn center is between 20 (exclusive) and 150 (inclusive) miles, add to list
    if min_distance_to_burn_center > 20 and min_distance_to_burn_center <= 150:
        non_burn_centers_30_150_miles_list.append(row_non_burn)

# Convert the list of Series (rows) into a DataFrame
df_non_burn_centers_30_150_miles = pd.DataFrame(non_burn_centers_30_150_miles_list)

print(f"Number of non-burn centers where the closest burn center is between 20 and 150 miles away: {len(df_non_burn_centers_30_150_miles)}")
df_non_burn_centers_30_150_miles.head()

Number of non-burn centers where the closest burn center is between 20 and 150 miles away: 206


,STATE_FULL,STATE,COUNTY,ADDRESS,CITY,ZIP_CODE,AHA_ID,HOSPITAL_NAME,TOTAL_BEDS,BURN_BEDS,...,ADULT_TRAUMA_L2,PEDS_TRAUMA_L1,PEDS_TRAUMA_L2,ABA_VERIFIED,TC_STATE_DESIGNATED,BC_STATE_DESIGNATED,PHONE,FULL_ADDRESS,Latitude,Longitude
5,Alabama,AL,Madison,101 Sivley Rd,Huntsville,35801,6530510.0,Huntsville Hospital,958,NaN,...,0.0,0.0,0.0,NaN,Yes,NaN,(256) 265-1000,"101 Sivley Rd, Huntsville, Alabama 35801",34.720467,-86.580963
7,Alabama,AL,Montgomery,2105 E South Blvd,Montgomery,36116,6530705.0,Baptist Medical Center South,492,NaN,...,1.0,0.0,0.0,NaN,Yes,NaN,(334) 288-2100,"2105 E South Blvd, Montgomery, Alabama 36116",32.327953,-86.277782
8,Arizona,AZ,Coconino,1200 N Beaver St,Flagstaff,86001,6860021.0,Flagstaff Medical Center,268,NaN,...,1.0,0.0,0.0,NaN,Yes,NaN,(928) 779-3366,"1200 N Beaver St, Flagstaff, Arizona 86001",35.208790,-111.644022
22,Arkansas,AR,Garland,300 Werner St,Hot Springs,71913,6710300.0,CHI St. Vincent Hospital Hot Springs,254,NaN,...,1.0,0.0,0.0,NaN,Yes,NaN,(501) 622-1000,"300 Werner St, Hot Springs, Arkansas 71913",34.467363,-93.067225
27,Arkansas,AR,Washington,3215 N Northhills Blvd,Fayetteville,72703,6710195.0,Washington Regional Medical Center,342,NaN,...,1.0,0.0,0.0,NaN,Yes,NaN,(479) 463-1000,"3215 N Northhills Blvd, Fayetteville, Arkansas...",36.110236,-94.159764


In [29]:
import folium

# Initialize a Folium map centered around the average coordinates of the burn centers
# Using a default location if df_burn_center is empty or has no coordinates
if not df_burn_center.empty and 'Latitude' in df_burn_center.columns and 'Longitude' in df_burn_center.columns:
    map_center = [df_burn_center['Latitude'].mean(), df_burn_center['Longitude'].mean()]
else:
    map_center = [39.8283, -98.5795] # Geographic center of the contiguous United States

m_30_150 = folium.Map(location=map_center, zoom_start=4)
print("Folium map initialized for 30-150 mile range visualization.")

Folium map initialized for 30-150 mile range visualization.


In [30]:
for index, row in df_burn_center.iterrows():
    # Create popup content for burn centers
    popup_content = f"<b>Burn Center:</b> {row['HOSPITAL_NAME']}<br>"
    popup_content += f"<b>Address:</b> {row['FULL_ADDRESS']}<br>"
    popup_content += f"<b>Burn Beds:</b> {row['BURN_BEDS'] if pd.notna(row['BURN_BEDS']) else 'N/A'}<br>"
    popup_content += f"<b>ACS Verified:</b> {row['ACS_VERIFIED']}<br>"
    popup_content += f"<b>ABA Verified:</b> {row['ABA_VERIFIED']}"

    # Add marker for burn center
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(popup_content, max_width=300),
        tooltip=row['HOSPITAL_NAME'],
        icon=folium.Icon(color='red', icon='fire', prefix='fa') # Distinct icon for burn centers
    ).add_to(m_30_150)

    # Add a 30-mile radius circle around burn centers (1 mile = 1609.34 meters)
    folium.Circle(
        location=[row['Latitude'], row['Longitude']],
        radius=30 * 1609.34, # Radius in meters
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.1
    ).add_to(m_30_150)


for index, row in df_non_burn_centers_30_150_miles.iterrows():
    # Create popup content for non-burn centers in the 30-150 mile range
    popup_content = f"<b>Non-Burn Center (30-150 mi):</b> {row['HOSPITAL_NAME']}<br>"
    popup_content += f"<b>Address:</b> {row['FULL_ADDRESS']}"

    # Add marker for nearby non-burn center with a different color
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(popup_content, max_width=300),
        tooltip=row['HOSPITAL_NAME'],
        icon=folium.Icon(color='blue', icon='hospital', prefix='fa') # Distinct icon for non-burn centers
    ).add_to(m_30_150)

# Display the map
m_30_150

### 6.3 Proximity Category: Beyond 150 Miles
Identify remote non-burn centers that are more than 150 miles away from any specialized burn center. Telemedicine and potential expansion of burn care facilities are crucial here.

In [31]:
non_burn_centers_150_miles_plus_list = []

for index_non_burn, row_non_burn in df_non_burn_center.iterrows():
    non_burn_center_coords = (row_non_burn['Latitude'], row_non_burn['Longitude'])
    min_distance_to_burn_center = float('inf') # Initialize with a very large distance

    # Skip if non-burn center coordinates are not valid
    if pd.isna(non_burn_center_coords[0]) or pd.isna(non_burn_center_coords[1]):
        continue

    for index_burn, row_burn in df_burn_center.iterrows():
        burn_center_coords = (row_burn['Latitude'], row_burn['Longitude'])

        # Ensure both coordinates are not None/NaN before calculating distance
        if pd.notna(burn_center_coords[0]) and pd.notna(burn_center_coords[1]):
            distance_miles = geodesic(non_burn_center_coords, burn_center_coords).miles
            min_distance_to_burn_center = min(min_distance_to_burn_center, distance_miles)

    # If the minimum distance to *any* burn center is greater than 150 miles, add to list
    if min_distance_to_burn_center > 150:
        non_burn_centers_150_miles_plus_list.append(row_non_burn)

# Convert the list of Series (rows) into a DataFrame
non_burn_centers_150_miles_plus = pd.DataFrame(non_burn_centers_150_miles_plus_list)

print(f"Number of non-burn centers where the closest burn center is more than 150 miles away: {len(non_burn_centers_150_miles_plus)}")
non_burn_centers_150_miles_plus.head()

Number of non-burn centers where the closest burn center is more than 150 miles away: 29


,STATE_FULL,STATE,COUNTY,ADDRESS,CITY,ZIP_CODE,AHA_ID,HOSPITAL_NAME,TOTAL_BEDS,BURN_BEDS,...,ADULT_TRAUMA_L2,PEDS_TRAUMA_L1,PEDS_TRAUMA_L2,ABA_VERIFIED,TC_STATE_DESIGNATED,BC_STATE_DESIGNATED,PHONE,FULL_ADDRESS,Latitude,Longitude
0,Alaska,AK,Anchorage,4315 Diplomacy Dr,Anchorage,99508,6940010.0,Alaska Native Medical Center,173,NaN,...,1.0,0.0,1.0,NaN,Yes,NaN,(907) 563-2662,"4315 Diplomacy Dr, Anchorage, Alaska 99508",61.182772,-149.800177
1,Alaska,AK,Anchorage,3200 Providence Dr,Anchorage,99508,6940020.0,Providence Alaska Medical Center/Children's Ho...,401,NaN,...,1.0,0.0,1.0,NaN,Yes,NaN,(907) 562-2211,"3200 Providence Dr, Anchorage, Alaska 99508",61.187168,-149.819439
2,Alabama,AL,Houston,1108 Ross Clark Cir,Dothan,36301,6530373.0,Southeast Alabama Medical Center,387,NaN,...,1.0,0.0,0.0,NaN,Yes,NaN,(334) 793-8111,"1108 Ross Clark Cir, Dothan, Alabama 36301",31.214885,-85.361261
107,Colorado,CO,Mesa,2635 N 7th St,Grand Junction,81501,6840625.0,St. Mary's Hospital and Medical Center,310,NaN,...,1.0,0.0,0.0,NaN,Yes,NaN,(970) 298-2273,"2635 N 7th St, Grand Junction, Colorado 81501",39.090295,-108.563204
128,Florida,FL,Bay,615 N Bonita Ave,Panama City,32401,6390775.0,Ascension Sacred Heart Bay,202,NaN,...,1.0,0.0,0.0,NaN,Yes,NaN,(850) 769-1511,"615 N Bonita Ave, Panama City, Florida 32401",30.159618,-85.648839


In [32]:
# Initialize a Folium map centered around the average coordinates of the burn centers
# Using a default location if df_burn_center is empty or has no coordinates
if not df_burn_center.empty and 'Latitude' in df_burn_center.columns and 'Longitude' in df_burn_center.columns:
    map_center = [df_burn_center['Latitude'].mean(), df_burn_center['Longitude'].mean()]
else:
    map_center = [39.8283, -98.5795] # Geographic center of the contiguous United States

m_150 = folium.Map(location=map_center, zoom_start=4)
print("Folium map initialized for 150 mile+ range visualization.")

Folium map initialized for 150 mile+ range visualization.


In [33]:
for index, row in df_burn_center.iterrows():
    # Create popup content for burn centers
    popup_content = f"<b>Burn Center:</b> {row['HOSPITAL_NAME']}<br>"
    popup_content += f"<b>Address:</b> {row['FULL_ADDRESS']}<br>"
    popup_content += f"<b>Burn Beds:</b> {row['BURN_BEDS'] if pd.notna(row['BURN_BEDS']) else 'N/A'}<br>"
    popup_content += f"<b>ACS Verified:</b> {row['ACS_VERIFIED']}<br>"
    popup_content += f"<b>ABA Verified:</b> {row['ABA_VERIFIED']}"

    # Add marker for burn center
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(popup_content, max_width=300),
        tooltip=row['HOSPITAL_NAME'],
        icon=folium.Icon(color='red', icon='fire', prefix='fa') # Distinct icon for burn centers
    ).add_to(m_150)

    # Add a 30-mile radius circle around burn centers (1 mile = 1609.34 meters)
    folium.Circle(
        location=[row['Latitude'], row['Longitude']],
        radius=30 * 1609.34, # Radius in meters
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.1
    ).add_to(m_150)


for index, row in non_burn_centers_150_miles_plus.iterrows():
    # Create popup content for non-burn centers > 150 miles away
    popup_content = f"<b>Non-Burn Center (>150 mi):</b> {row['HOSPITAL_NAME']}<br>"
    popup_content += f"<b>Address:</b> {row['FULL_ADDRESS']}"

    # Add marker for nearby non-burn center with a different color
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(popup_content, max_width=300),
        tooltip=row['HOSPITAL_NAME'],
        icon=folium.Icon(color='blue', icon='hospital', prefix='fa') # Distinct icon for non-burn centers
    ).add_to(m_150)

# Display the map
m_150

## 7. Comprehensive Analysis and Implications for Burn System Design

The geographic distribution of specialized burn centers relative to general acute care facilities highlights the critical importance of regionalized trauma systems. Given the highly specialized and resource-intensive nature of burn care, a "hub-and-spoke" model is typically employed. In this model, specialized regional burn centers (hubs) receive high-acuity patients from surrounding non-burn acute care facilities (spokes). The findings of this spatial analysis can be contextualized within three distinct pre-hospital transport and triage modalities:

### 7.1 The Immediate Catchment Zone (Ground Transport Radius: ≤ 30 Miles)
*   **Findings:** A total of 267 non-burn centers were identified within a 30-mile radius of a verified or designated burn facility.
*   **Clinical Implications:** This zone represents the immediate geographic catchment area where ground Emergency Medical Services (EMS) are highly effective and rapid transfer is feasible. Non-burn centers within this radius serve a crucial role in initial stabilization (e.g., airway management, primary fluid resuscitation) before rapid ground transfer. Furthermore, these proximate facilities offer robust opportunities for step-down care, post-acute rehabilitation, and seamless resource sharing, optimizing the utilization of high-acuity intensive care beds at the specialized hubs.

### 7.2 The Intermediate Zone (Aeromedical Transport Radius: 30 - 150 Miles)
*   **Findings:** 206 non-burn centers fall within the 20 to 150-mile intermediate range.
*   **Clinical Implications:** This distance typically exceeds optimal ground transport times for highly critical burns but falls squarely within the operational radius of Helicopter Emergency Medical Services (HEMS) and regional fixed-wing transport. Centers in this geographic band must maintain rigorous Advanced Burn Life Support (ABLS) capabilities to manage the crucial "golden hours" of a major burn prior to aeromedical evacuation. Establishing formalized referral networks, integrated transport protocols, and clear triage criteria is vital to prevent delays in definitive care.

### 7.3 The Remote Zone (Tele-Burn and Infrastructure Expansion: > 150 Miles)
*   **Findings:** 29 non-burn centers are situated more than 150 miles from the nearest specialized burn facility.
*   **Clinical Implications:** This remote classification highlights significant spatial disparities in healthcare access. In these regions, prolonged transport times present a major risk to severe burn outcomes. Consequently, these areas rely heavily on "Tele-Burn" infrastructure—utilizing telemedicine for expert remote triage, fluid resuscitation guidance, and wound assessment. From a public health policy perspective, these remote clusters represent primary candidates for targeted ABLS training and the potential strategic expansion of new burn care infrastructure to mitigate geographic inequity.

### Academic & Clinical References
*   [Helicopter Transport and Burn Patients (JAMA)](https://jamanetwork.com/journals/jama/fullarticle/184780)
*   [Pre-hospital Management, Transportation, and Emergency Care](https://plasticsurgerykey.com/pre-hospital-management-transportation-and-emergency-care/)